# Geocoding from Text

Converts address or place-name text into latitude and longitude using the free Nominatim geocoder (OpenStreetMap). Adds `Latitude#number#hidden`, `Longitude#number#hidden`, and `Geocode_Quality` columns, then shows a preview map.

Nominatim requires 1 second between requests. For large surveys (>500 rows) this will take a few minutes.

In [ ]:
# ── Colab bootstrap (no-op on Binder / JupyterHub) ────────────────────────
import os, sys
if "google.colab" in sys.modules:
    if not os.path.isfile("/tmp/suave-nb/helpers/suave_utils.py"):
        import subprocess
        _r = subprocess.run(
            ["git", "clone", "--depth=1", "https://github.com/izaslavsky/suave-notebooks.git", "/tmp/suave-nb"],
            capture_output=True, text=True
        )
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

if not su.ENV.colab:
    # Binder / JupyterHub: parameters load automatically
    _p = su.load_params()
    if _p is None:
        raise RuntimeError('No SuAVE params. Open via SuAVEDispatch.ipynb first.')
else:
    # Colab: use Drive (automatic) or enter credentials below
    SUAVE_TOKEN = ''   # @param {type:"string"}
    SUAVE_HOST  = ''   # @param {type:"string"}
    _p = su.load_params(token=SUAVE_TOKEN, host=SUAVE_HOST)

user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'
_prefix       = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/')
full_notebook_url = _prefix.rstrip('/') + '/lab/tree/SuAVEDispatch.ipynb'

localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images

absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)

In [ ]:
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint

import ipywidgets as widgets
import pandas as pd
import numpy as np
import time

## 1. Load data and select address column

In [ ]:
df = panellibs.extract_data(absolutePath + csv_file)
print(f"Loaded {len(df)} rows")

text_cols = [c for c in df.columns if '#number' not in c and '#img' not in c
             and '#hidden' not in c and '#date' not in c]

col_selector = widgets.Dropdown(
    options=text_cols, description='Address column:',
    layout=widgets.Layout(width='60%')
)
country_hint = widgets.Text(
    value='', placeholder='e.g. USA  (optional, improves accuracy)',
    description='Country hint:',
    layout=widgets.Layout(width='60%')
)
display(col_selector, country_hint)

## 2. Geocode addresses

In [ ]:
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
from tqdm.auto import tqdm

col    = col_selector.value
hint   = country_hint.value.strip()
geolocator = Nominatim(user_agent='suave-notebooks-geocoder')

lats, lons, qualities = [], [], []
cache = {}

for addr in tqdm(df[col], desc='Geocoding'):
    if not addr or pd.isna(addr) or str(addr).strip() == '':
        lats.append(None); lons.append(None); qualities.append('missing')
        continue

    query = str(addr).strip()
    if hint:
        query = f"{query}, {hint}"

    if query in cache:
        lats.append(cache[query][0])
        lons.append(cache[query][1])
        qualities.append(cache[query][2])
        continue

    try:
        time.sleep(1.1)  # Nominatim rate limit: 1 req/sec
        loc = geolocator.geocode(query, timeout=10)
        if loc:
            lat, lon = round(loc.latitude, 6), round(loc.longitude, 6)
            quality = 'ok'
        else:
            lat, lon, quality = None, None, 'not_found'
    except GeocoderTimedOut:
        lat, lon, quality = None, None, 'timeout'
    except Exception as e:
        lat, lon, quality = None, None, 'error'

    cache[query] = (lat, lon, quality)
    lats.append(lat); lons.append(lon); qualities.append(quality)

df['Latitude#number#hidden']  = lats
df['Longitude#number#hidden'] = lons
df['Geocode_Quality']         = qualities

found = sum(1 for q in qualities if q == 'ok')
printmd(f"**Geocoded {found}/{len(df)} rows successfully.**")
print(df['Geocode_Quality'].value_counts().to_string())

## 3. Preview map

In [ ]:
import folium

mapped = df.dropna(subset=['Latitude#number#hidden', 'Longitude#number#hidden'])
if len(mapped) == 0:
    printmd('**No rows were geocoded successfully.**')
else:
    center_lat = mapped['Latitude#number#hidden'].astype(float).mean()
    center_lon = mapped['Longitude#number#hidden'].astype(float).mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=4)

    for _, row in mapped.iterrows():
        folium.CircleMarker(
            location=[float(row['Latitude#number#hidden']), float(row['Longitude#number#hidden'])],
            radius=5, color='#3498db', fill=True, fill_opacity=0.7,
            popup=str(row.get(col, ''))
        ).add_to(m)

    display(m)

## 4. Save and publish to SuAVE

In [ ]:
new_file = suaveint.save_csv_file(df, absolutePath, csv_file)

In [ ]:
input_text = widgets.Text(placeholder='Enter survey name...')
output_text = widgets.Text()

def _bind(sender):
    output_text.value = input_text.value

input_text.on_submit(_bind)
printmd("**Enter survey name, press Enter, then run the next cell:**")
display(input_text, output_text)

In [ ]:
survey_name = output_text.value
suaveint.create_survey(survey_url, new_file, survey_name, dzc_file, user, csv_file, view, views)